In [1]:
# IMPORT & LOAD DATA
import pandas as pd

# Load cleaned data
df = pd.read_csv("D:/customer-segmentation-unsupervised/data/processed/clean_data.csv")

# Convert datetime again (safe)
df["event_time"] = pd.to_datetime(df["event_time"])

print("Data Loaded ")
df.head()

Data Loaded 


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,date,hour,day_of_week
0,2019-11-01 00:00:00+00:00,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33,2019-11-01,0,Friday
1,2019-11-01 00:00:00+00:00,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283,2019-11-01,0,Friday
2,2019-11-01 00:00:01+00:00,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387,2019-11-01,0,Friday
3,2019-11-01 00:00:01+00:00,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f,2019-11-01,0,Friday
4,2019-11-01 00:00:01+00:00,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2,2019-11-01,0,Friday


In [2]:
# FILTER PURCHASE DATA
# Filter purchase events
purchase_df = df[df["event_type"] == "purchase"]

print("Purchase Data Shape:", purchase_df.shape)

Purchase Data Shape: (9595, 12)


In [3]:
# CREATE RFM FEATURES
# Reference date (latest date in dataset)
snapshot_date = df["event_time"].max()

# Create RFM table
rfm = purchase_df.groupby("user_id").agg({
    "event_time": lambda x: (snapshot_date - x.max()).days,  # Recency
    "user_id": "count",  # Frequency
    "price": "sum"       # Monetary
})

# Rename columns
rfm.columns = ["Recency", "Frequency", "Monetary"]

rfm.head()

,Recency,Frequency,Monetary
user_id,,,
356520186,0,1,33.45
397023870,0,1,244.28
486999716,0,1,242.72
509881222,0,1,200.52
512364896,0,5,708.01


In [4]:
# ADD BEHAVIORAL FEATURES
# Total views
views = df[df["event_type"] == "view"].groupby("user_id").size()

# Total carts
carts = df[df["event_type"] == "cart"].groupby("user_id").size()

# Add to RFM
rfm["Views"] = views
rfm["Carts"] = carts

# Fill missing values
rfm = rfm.fillna(0)

rfm.head()

,Recency,Frequency,Monetary,Views,Carts
user_id,,,,,
356520186,0,1,33.45,5.0,0.0
397023870,0,1,244.28,3.0,0.0
486999716,0,1,242.72,3.0,0.0
509881222,0,1,200.52,55.0,0.0
512364896,0,5,708.01,16.0,3.0


In [5]:
# ADVANCED FEATURES
# Purchase ratio (conversion rate)
rfm["Purchase_Ratio"] = rfm["Frequency"] / (rfm["Views"] + 1)

# Average spending per purchase
rfm["Avg_Spend"] = rfm["Monetary"] / (rfm["Frequency"] + 1)

rfm.head()

,Recency,Frequency,Monetary,Views,Carts,Purchase_Ratio,Avg_Spend
user_id,,,,,,,
356520186,0,1,33.45,5.0,0.0,0.166667,16.725000
397023870,0,1,244.28,3.0,0.0,0.250000,122.140000
486999716,0,1,242.72,3.0,0.0,0.250000,121.360000
509881222,0,1,200.52,55.0,0.0,0.017857,100.260000
512364896,0,5,708.01,16.0,3.0,0.294118,118.001667


In [6]:
# BASIC
print(rfm.describe())
print("Total Customers:", rfm.shape[0])

       Recency    Frequency      Monetary        Views        Carts  \
count   7389.0  7389.000000   7389.000000  7389.000000  7389.000000   
mean       0.0     1.298552    392.153393     7.622953     0.669238   
std        0.0     0.921190    612.562163    10.573034     1.368966   
min        0.0     1.000000      1.000000     0.000000     0.000000   
25%        0.0     1.000000     83.660000     2.000000     0.000000   
50%        0.0     1.000000    193.020000     4.000000     0.000000   
75%        0.0     1.000000    458.280000     9.000000     1.000000   
max        0.0    17.000000  16222.800000   291.000000    29.000000   

       Purchase_Ratio    Avg_Spend  
count     7389.000000  7389.000000  
mean         0.246619   159.129637  
std          0.146287   186.379212  
min          0.003425     0.500000  
25%          0.125000    38.600000  
50%          0.250000    89.590000  
75%          0.333333   191.564286  
max          1.000000  1595.753333  
Total Customers: 7389


In [7]:
# SAVE FINAL DATA
rfm.to_csv("D:/customer-segmentation-unsupervised/data/processed/rfm_data.csv")

print("RFM Data Saved Successfully ")

RFM Data Saved Successfully 
